In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import warnings

warnings.filterwarnings('ignore')

In [3]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
original = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [4]:
train['Churn'] = train['Churn'].map({'Yes': 1, 'No': 0}).astype(int)
original['Churn'] = original['Churn'].map({'Yes': 1, 'No': 0}).astype(int)

original['TotalCharges'] = pd.to_numeric(original['TotalCharges'], errors='coerce')
original['TotalCharges'].fillna(original['TotalCharges'].median(), inplace=True)
# original['TotalCharges'].fillna(original['MonthlyCharges'] * original['tenure'], inplace=True)

In [5]:
original.drop('customerID', axis=1, inplace=True)
train.drop('id', axis=1, inplace=True)
test.drop('id', axis=1, inplace=True)

In [6]:
cats = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod'
]

nums = ['tenure', 'MonthlyCharges', 'TotalCharges']

target = 'Churn'

new_nums = []
nums_as_cat = []

In [7]:
for col in nums:
    freq = pd.concat([train[col], original[col], test[col]]).value_counts(normalize=True)
    for df in [train, test, original]:
        df[f'Freq{col}'] = df[col].map(freq).fillna(0).astype('float32')
    new_nums.append(f'Freq{col}')

In [8]:
for col in cats:

    churn_mean = original.groupby(col)[target].mean()
    churn_std = original.groupby(col)[target].std()

    train[f"{col}ChurnMean"] = train[col].map(churn_mean)
    test[f"{col}ChurnMean"] = test[col].map(churn_mean)

    train[f"{col}ChurnStd"] = train[col].map(churn_std)
    test[f"{col}ChurnStd"] = test[col].map(churn_std)

In [9]:
q1 = train["MonthlyCharges"].quantile(0.25)
q3 = train["MonthlyCharges"].quantile(0.75)

for df in [train, test]:

    df["ChargeOutlier"] = (
        (df["MonthlyCharges"] < q1) |
        (df["MonthlyCharges"] > q3)
    ).astype(int)

    df['MCLog'] = np.log1p(df['MonthlyCharges'])
    df['TCLog'] = np.log1p(df['TotalCharges'])

    df['MCSqueezed'] = 1 / (df['MonthlyCharges'] + 1)
    df['TCSqueezed'] = 1 / (df['TotalCharges'] + 1)

    df["TenureLog"] = np.log1p(df["tenure"])
    df['TenureSqueezed'] = 1 / (df['tenure'] + 1)

    df["AvgMonthlySpend"] = df["TotalCharges"] / (df["tenure"] + 1)
    df["SpendDeviation"] = df["MonthlyCharges"] - df["AvgMonthlySpend"]
    df["MonthlyToTotalRatio"] = df["MonthlyCharges"] / (df["TotalCharges"] + 1)

    df["IsMonthToMonth"] = (df["Contract"] == "Month-to-month").astype(int)

    df["IsElectronicCheck"] = (df["PaymentMethod"] == "Electronic check").astype(int)
    df["AutoPayment"] = df["PaymentMethod"].str.contains("automatic").astype(int)

    df["ServiceCount"] = (
        (df["OnlineSecurity"] == "Yes").astype(int) +
        (df["OnlineBackup"] == "Yes").astype(int) +
        (df["DeviceProtection"] == "Yes").astype(int) +
        (df["TechSupport"] == "Yes").astype(int) +
        (df["StreamingTV"] == "Yes").astype(int) +
        (df["StreamingMovies"] == "Yes").astype(int)
    )

    df["IsFiber"] = (df["InternetService"] == "Fiber optic").astype(int)

    df["FiberHighCharge"] = (
        (df["InternetService"] == "Fiber optic") &
        (df["MonthlyCharges"] > df["MonthlyCharges"].median())
    ).astype(int)

    df["FiberNoSupport"] = (
        (df["InternetService"] == "Fiber optic") &
        (df["TechSupport"] == "No")
    ).astype(int)

    df["EarlyTenure"] = (df["tenure"] < 12).astype(int)
    df["HighChargeEarly"] = (
        (df["tenure"] < 12) &
        (df["MonthlyCharges"] > df["MonthlyCharges"].median())
    ).astype(int)

    df["TenureBucket"] = pd.cut(
        df["tenure"],
        bins=[0, 6, 12, 24, 48, 72],
        labels=False
    )

    df["VeryEarly"] = (df["tenure"] < 6).astype(int)

    df["LifetimeValue"] = df["MonthlyCharges"] * df["tenure"]

    df["ChargeRank"] = df["MonthlyCharges"].rank(pct=True)
    df["TenureRank"] = df["tenure"].rank(pct=True)

new_nums = [col for col in df.columns.tolist() if col not in (nums + cats + [target])]

In [10]:
for col in cats + nums:
    tmp = original.groupby(col)[target].mean()
    name = f"OrigP{col}"
    train = train.merge(tmp.rename(name), on=col, how="left")
    test = test.merge(tmp.rename(name), on=col, how="left")
    for df in [train, test]:
        df[name] = df[name].fillna(0.5).astype('float32')
    new_nums.append(name)

In [11]:
X, y = train.drop(target, axis=1), train[target]

In [12]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

label_cols = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling'
]

onehot_cols = ['PaymentMethod']

ord_mapping = {
    'Contract': ['Two year', 'One year', 'Month-to-month'],
    'MultipleLines': ['No phone service', 'No', 'Yes'],
    'InternetService': ['No', 'DSL', 'Fiber optic'],
    'OnlineSecurity': ['No internet service', 'No', 'Yes'],
    'OnlineBackup': ['No internet service', 'No', 'Yes'],
    'DeviceProtection': ['No internet service', 'No', 'Yes'],
    'TechSupport': ['No internet service', 'No', 'Yes'],
    'StreamingTV': ['No internet service', 'No', 'Yes'],
    'StreamingMovies': ['No internet service', 'No', 'Yes'],
    'Partner': ['No', 'Yes'],
    'Dependents': ['No', 'Yes'],
    'PhoneService': ['No', 'Yes'],
    'PaperlessBilling': ['No', 'Yes'],
    'gender': ['Male', 'Female']
}

ord_cols = list(ord_mapping.keys())

preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(drop='first', sparse_output=False), onehot_cols),
        ('ordinal', OrdinalEncoder(categories=[ord_mapping[col] for col in ord_cols]), ord_cols)
    ],
    remainder='passthrough'
)

pipeline = Pipeline([
    ('preprocessor', preprocessor)
])

pipeline.set_output(transform="pandas")

X_train_encoded = pipeline.fit_transform(X)
X_test_encoded = pipeline.transform(test)